# 05.2 Systematic Concurrency Experiments

Use this notebook for the formal experiment runs.

It automatically:
- rebuilds `library_sim` before every strategy,
- runs baseline, pessimistic, and optimistic tests,
- repeats each workload several times,
- saves every run to `library_results.test_run`,
- calculates averages at the end.

Keep `05.1_concurrency_tests.ipynb` as the development/debug notebook.


## 1. Imports


In [1]:
import mysql.connector
import threading
import time
import pandas as pd

from getpass import getpass


## 2. MySQL connection


In [2]:
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": getpass("MySQL password: ")
}

def get_connection(database=None):
    config = DB_CONFIG.copy()

    if database:
        config["database"] = database

    return mysql.connector.connect(**config)


## 3. Experiment plan

For each workload:

`total operations = threads × operations per thread`

The baseline uses the same total operations as pessimistic and optimistic.

`copy_ids = [1]` means all operations target the same copy, which creates high contention.
Later, use more copy IDs to test lower contention.


In [3]:
BATCH_NAME = "contention_comparison_v1"
REPEATS = 3

TEST_CONFIGS = [
    # LOW WORKLOAD
    {"workload": "Low", "threads": 5, "ops_per_thread": 5,
     "contention": "High", "copy_ids": [1]},

    {"workload": "Low", "threads": 5, "ops_per_thread": 5,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "Low", "threads": 5, "ops_per_thread": 5,
     "contention": "Low", "copy_ids": list(range(1, 26))},


    # MEDIUM WORKLOAD
    {"workload": "Medium", "threads": 20, "ops_per_thread": 5,
     "contention": "High", "copy_ids": [1]},

    {"workload": "Medium", "threads": 20, "ops_per_thread": 5,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "Medium", "threads": 20, "ops_per_thread": 5,
     "contention": "Low", "copy_ids": list(range(1, 26))},


    # HIGH WORKLOAD
    {"workload": "High", "threads": 50, "ops_per_thread": 20,
     "contention": "High", "copy_ids": [1]},

    {"workload": "High", "threads": 50, "ops_per_thread": 20,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "High", "threads": 50, "ops_per_thread": 20,
     "contention": "Low", "copy_ids": list(range(1, 26))},


    # STRESS WORKLOAD
    {"workload": "Stress", "threads": 50, "ops_per_thread": 100,
     "contention": "High", "copy_ids": [1]},

    {"workload": "Stress", "threads": 50, "ops_per_thread": 100,
     "contention": "Medium", "copy_ids": [1, 2, 3, 4, 5]},

    {"workload": "Stress", "threads": 50, "ops_per_thread": 100,
     "contention": "Low", "copy_ids": list(range(1, 26))}
]

for config in TEST_CONFIGS:
    total = config["threads"] * config["ops_per_thread"]

    print(
        f"{config['workload']} | "
        f"{config['contention']} contention | "
        f"{config['threads']} threads × "
        f"{config['ops_per_thread']} ops/thread = "
        f"{total:,} total operations | "
        f"{len(config['copy_ids'])} copy/copies"
    )


Low | High contention | 5 threads × 5 ops/thread = 25 total operations | 1 copy/copies
Low | Medium contention | 5 threads × 5 ops/thread = 25 total operations | 5 copy/copies
Low | Low contention | 5 threads × 5 ops/thread = 25 total operations | 25 copy/copies
Medium | High contention | 20 threads × 5 ops/thread = 100 total operations | 1 copy/copies
Medium | Medium contention | 20 threads × 5 ops/thread = 100 total operations | 5 copy/copies
Medium | Low contention | 20 threads × 5 ops/thread = 100 total operations | 25 copy/copies
High | High contention | 50 threads × 20 ops/thread = 1,000 total operations | 1 copy/copies
High | Medium contention | 50 threads × 20 ops/thread = 1,000 total operations | 5 copy/copies
High | Low contention | 50 threads × 20 ops/thread = 1,000 total operations | 25 copy/copies
Stress | High contention | 50 threads × 100 ops/thread = 5,000 total operations | 1 copy/copies
Stress | Medium contention | 50 threads × 100 ops/thread = 5,000 total operations 

## 4. Create permanent results table


In [4]:
def create_results_table():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("CREATE SCHEMA IF NOT EXISTS `library_results`")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS `library_results`.`test_run` (
            run_id INT NOT NULL AUTO_INCREMENT,
            batch_name VARCHAR(100) NOT NULL,
            workload VARCHAR(45) NOT NULL,
            strategy VARCHAR(20) NOT NULL,
            contention VARCHAR(20) NOT NULL,
            thread_count INT NOT NULL,
            operations_per_thread INT NOT NULL,
            total_operations INT NOT NULL,
            repeat_number INT NOT NULL,
            copy_count INT NOT NULL,
            runtime_seconds DECIMAL(14,6),
            successes INT,
            conflicts INT,
            deadlocks INT,
            timeouts INT,
            errors INT,
            attempt_throughput DECIMAL(14,3),
            successful_throughput DECIMAL(14,3),
            run_date DATETIME DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (run_id)
        )
    """)

    conn.commit()
    cursor.close()
    conn.close()

create_results_table()
print("Results table ready.")


Results table ready.


## 5. Rebuild `library_sim` before each strategy


In [5]:
def rebuild_library_sim():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("DROP SCHEMA IF EXISTS `library_sim`")
    cursor.execute("CREATE SCHEMA `library_sim`")

    tables = [
        "Book",
        "Book_copy",
        "Member",
        "Loan",
        "Transaction_log",
        "Conflict_log",
        "Deadlock_log"
    ]

    for table in tables:
        cursor.execute(
            f"CREATE TABLE `library_sim`.`{table}` "
            f"LIKE `library`.`{table}`"
        )

        cursor.execute(
            f"INSERT INTO `library_sim`.`{table}` "
            f"SELECT * FROM `library`.`{table}`"
        )

    cursor.execute("DELETE FROM `library_sim`.`Transaction_log`")
    cursor.execute("DELETE FROM `library_sim`.`Conflict_log`")
    cursor.execute("DELETE FROM `library_sim`.`Deadlock_log`")

    conn.commit()
    cursor.close()
    conn.close()


## 6. Checkout procedures


In [6]:
BASELINE_SQL = """
CREATE PROCEDURE checkout_baseline(IN p_copy_id INT)
BEGIN
    DECLARE v_version INT;

    START TRANSACTION;

    SELECT version
    INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id;

    INSERT INTO transaction_log
    (
        operation_type,
        timestamp,
        isolation_level,
        strategy,
        success_flag,
        conflict_flag,
        deadlock_flag
    )
    VALUES
    (
        'checkout',
        NOW(),
        'READ COMMITTED',
        'baseline',
        'TRUE',
        'FALSE',
        'FALSE'
    );

    COMMIT;
END
"""

PESSIMISTIC_SQL = """
CREATE PROCEDURE checkout_pessimistic(IN p_copy_id INT)
BEGIN
    DECLARE v_version INT;

    START TRANSACTION;

    SELECT version
    INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id
    FOR UPDATE;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id;

    INSERT INTO transaction_log
    (
        operation_type,
        timestamp,
        isolation_level,
        strategy,
        success_flag,
        conflict_flag,
        deadlock_flag
    )
    VALUES
    (
        'checkout',
        NOW(),
        'READ COMMITTED',
        'pessimistic',
        'TRUE',
        'FALSE',
        'FALSE'
    );

    COMMIT;
END
"""

OPTIMISTIC_SQL = """
CREATE PROCEDURE checkout_optimistic(
    IN p_copy_id INT,
    IN p_member_id INT,
    IN p_isolation VARCHAR(45)
)
BEGIN
    DECLARE v_version INT;

    SELECT version
    INTO v_version
    FROM book_copy
    WHERE copy_id = p_copy_id;

    UPDATE book_copy
    SET version = version + 1
    WHERE copy_id = p_copy_id
      AND version = v_version;

    IF ROW_COUNT() = 0 THEN

        INSERT INTO conflict_log
        (
            old_version,
            attempted_version,
            timestamp
        )
        VALUES
        (
            v_version,
            v_version + 1,
            NOW()
        );

        INSERT INTO transaction_log
        (
            operation_type,
            timestamp,
            isolation_level,
            strategy,
            success_flag,
            conflict_flag,
            deadlock_flag
        )
        VALUES
        (
            'checkout',
            NOW(),
            p_isolation,
            'optimistic',
            'FALSE',
            'TRUE',
            'FALSE'
        );

    ELSE

        INSERT INTO transaction_log
        (
            operation_type,
            timestamp,
            isolation_level,
            strategy,
            success_flag,
            conflict_flag,
            deadlock_flag
        )
        VALUES
        (
            'checkout',
            NOW(),
            p_isolation,
            'optimistic',
            'TRUE',
            'FALSE',
            'FALSE'
        );

    END IF;

    COMMIT;
END
"""

def create_checkout_procedures():
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    procedures = [
        ("checkout_baseline", BASELINE_SQL),
        ("checkout_pessimistic", PESSIMISTIC_SQL),
        ("checkout_optimistic", OPTIMISTIC_SQL)
    ]

    for name, sql in procedures:
        cursor.execute(f"DROP PROCEDURE IF EXISTS `{name}`")
        cursor.execute(sql)

    cursor.close()
    conn.close()


## 7. Read actual successes and conflicts


In [7]:
def get_db_metrics(strategy):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s
          AND success_flag = 'TRUE'
    """, (strategy,))

    successes = cursor.fetchone()[0]

    cursor.execute("""
        SELECT COUNT(*)
        FROM transaction_log
        WHERE strategy = %s
          AND conflict_flag = 'TRUE'
    """, (strategy,))

    conflicts = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    return successes, conflicts


## 8. Baseline runner


In [8]:
def run_baseline(total_operations, copy_ids):
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    start_time = time.perf_counter()

    for i in range(total_operations):
        copy_id = copy_ids[i % len(copy_ids)]
        cursor.callproc("checkout_baseline", [copy_id])

    runtime = time.perf_counter() - start_time

    cursor.close()
    conn.close()

    return runtime


## 9. Pessimistic runner


In [9]:
metric_lock = threading.Lock()

def run_pessimistic(num_threads, operations_per_thread, copy_ids):
    metrics = {
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }

    def worker(thread_id):
        conn = get_connection("library_sim")
        cursor = conn.cursor()

        for operation in range(operations_per_thread):
            copy_id = copy_ids[
                (thread_id + operation) % len(copy_ids)
            ]

            try:
                cursor.callproc(
                    "checkout_pessimistic",
                    [copy_id]
                )

            except Exception as e:
                error_text = str(e)

                with metric_lock:
                    metrics["errors"] += 1

                    if "1205" in error_text:
                        metrics["timeouts"] += 1

                    if "1213" in error_text:
                        metrics["deadlocks"] += 1

                conn.rollback()

        cursor.close()
        conn.close()

    threads = []
    start_time = time.perf_counter()

    for thread_id in range(num_threads):
        thread = threading.Thread(
            target=worker,
            args=(thread_id,)
        )

        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    runtime = time.perf_counter() - start_time

    return runtime, metrics


## 10. Optimistic runner


In [10]:
def run_optimistic(num_threads, operations_per_thread, copy_ids):
    metrics = {
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }

    def worker(thread_id):
        conn = get_connection("library_sim")
        cursor = conn.cursor()

        for operation in range(operations_per_thread):
            copy_id = copy_ids[
                (thread_id + operation) % len(copy_ids)
            ]

            try:
                cursor.callproc(
                    "checkout_optimistic",
                    [
                        copy_id,
                        thread_id,
                        "READ COMMITTED"
                    ]
                )

                conn.commit()

            except Exception as e:
                error_text = str(e)

                with metric_lock:
                    metrics["errors"] += 1

                    if "1205" in error_text:
                        metrics["timeouts"] += 1

                    if "1213" in error_text:
                        metrics["deadlocks"] += 1

                conn.rollback()

        cursor.close()
        conn.close()

    threads = []
    start_time = time.perf_counter()

    for thread_id in range(num_threads):
        thread = threading.Thread(
            target=worker,
            args=(thread_id,)
        )

        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    runtime = time.perf_counter() - start_time

    return runtime, metrics


## 11. Save each run permanently


In [11]:
def save_result(
    config,
    strategy,
    repeat_number,
    runtime,
    successes,
    conflicts,
    metrics
):
    total_operations = (
        config["threads"]
        * config["ops_per_thread"]
    )

    attempt_throughput = (
        total_operations / runtime
        if runtime > 0
        else 0
    )

    successful_throughput = (
        successes / runtime
        if runtime > 0
        else 0
    )

    conn = get_connection("library_results")
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO test_run
        (
            batch_name,
            workload,
            strategy,
            contention,
            thread_count,
            operations_per_thread,
            total_operations,
            repeat_number,
            copy_count,
            runtime_seconds,
            successes,
            conflicts,
            deadlocks,
            timeouts,
            errors,
            attempt_throughput,
            successful_throughput
        )
        VALUES
        (
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s
        )
    """, (
        BATCH_NAME,
        config["workload"],
        strategy,
        config["contention"],
        config["threads"],
        config["ops_per_thread"],
        total_operations,
        repeat_number,
        len(config["copy_ids"]),
        runtime,
        successes,
        conflicts,
        metrics["deadlocks"],
        metrics["timeouts"],
        metrics["errors"],
        attempt_throughput,
        successful_throughput
    ))

    conn.commit()
    cursor.close()
    conn.close()


## 12. Run the full experiment matrix

This is the main experiment cell.

For every workload and repeat:
1. fresh clone → baseline → save,
2. fresh clone → pessimistic → save,
3. fresh clone → optimistic → save.


In [12]:
for config in TEST_CONFIGS:
    total_operations = (
        config["threads"]
        * config["ops_per_thread"]
    )

    print("\n" + "=" * 80)
    print(
        f"{config['workload']} | "
        f"{config['contention']} contention | "
        f"{config['threads']} threads | "
        f"{config['ops_per_thread']} ops/thread | "
        f"{total_operations:,} total operations | "
        f"{len(config['copy_ids'])} copy/copies"
    )
    print("=" * 80)

    for repeat in range(1, REPEATS + 1):
        print(f"\nRepeat {repeat} of {REPEATS}")

        # BASELINE
        print("  Baseline...")
        rebuild_library_sim()
        create_checkout_procedures()

        baseline_time = run_baseline(
            total_operations,
            config["copy_ids"]
        )

        baseline_successes, baseline_conflicts = (
            get_db_metrics("baseline")
        )

        save_result(
            config,
            "baseline",
            repeat,
            baseline_time,
            baseline_successes,
            baseline_conflicts,
            {
                "deadlocks": 0,
                "timeouts": 0,
                "errors": 0
            }
        )

        # PESSIMISTIC
        print("  Pessimistic...")
        rebuild_library_sim()
        create_checkout_procedures()

        pess_time, pess_metrics = run_pessimistic(
            config["threads"],
            config["ops_per_thread"],
            config["copy_ids"]
        )

        pess_successes, pess_conflicts = (
            get_db_metrics("pessimistic")
        )

        save_result(
            config,
            "pessimistic",
            repeat,
            pess_time,
            pess_successes,
            pess_conflicts,
            pess_metrics
        )

        # OPTIMISTIC
        print("  Optimistic...")
        rebuild_library_sim()
        create_checkout_procedures()

        opt_time, opt_metrics = run_optimistic(
            config["threads"],
            config["ops_per_thread"],
            config["copy_ids"]
        )

        opt_successes, opt_conflicts = (
            get_db_metrics("optimistic")
        )

        save_result(
            config,
            "optimistic",
            repeat,
            opt_time,
            opt_successes,
            opt_conflicts,
            opt_metrics
        )

print("\nAll systematic experiment runs finished.")



Low | High contention | 5 threads | 5 ops/thread | 25 total operations | 1 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 3 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Low | Medium contention | 5 threads | 5 ops/thread | 25 total operations | 5 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 3 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Low | Low contention | 5 threads | 5 ops/thread | 25 total operations | 25 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 3 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Medium | High contention | 20 threads | 5 ops/thread | 100 total operations | 1 copy/copies

Repeat 1 of 3
  Baseline...
  Pessimistic...
  Optimistic...

Repeat 2 of 3
  Base

## 13. View all individual runs for this batch


In [13]:
conn = get_connection("library_results")

runs_df = pd.read_sql(
    """
    SELECT *
    FROM test_run
    WHERE batch_name = %s
    ORDER BY
        total_operations,
        contention,
        repeat_number,
        strategy
    """,
    conn,
    params=(BATCH_NAME,)
)

conn.close()

runs_df


/var/folders/qy/yl4hyv4s5kz52wg1qlh36y_w0000gn/T/ipykernel_7299/527018973.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  runs_df = pd.read_sql(


,run_id,batch_name,workload,strategy,contention,thread_count,operations_per_thread,total_operations,repeat_number,copy_count,runtime_seconds,successes,conflicts,deadlocks,timeouts,errors,attempt_throughput,successful_throughput,run_date
0,1,contention_comparison_v1,Low,baseline,High,5,5,25,1,1,0.007014,25,0,0,0,0,3564.067,3564.067,2026-11-04 11:05:35
1,3,contention_comparison_v1,Low,optimistic,High,5,5,25,1,1,0.006504,17,8,0,0,0,3843.690,2613.709,2026-11-04 11:05:35
2,2,contention_comparison_v1,Low,pessimistic,High,5,5,25,1,1,0.006532,25,0,0,0,0,3827.214,3827.214,2026-11-04 11:05:35
3,4,contention_comparison_v1,Low,baseline,High,5,5,25,2,1,0.007474,25,0,0,0,0,3344.761,3344.761,2026-11-04 11:05:35
4,6,contention_comparison_v1,Low,optimistic,High,5,5,25,2,1,0.005648,14,11,0,0,0,4426.411,2478.790,2026-11-04 11:05:36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,96,contention_comparison_v1,Stress,optimistic,Medium,50,100,5000,2,5,0.609234,4194,806,0,0,0,8207.031,6884.057,2026-11-04 11:06:01
104,95,contention_comparison_v1,Stress,pessimistic,Medium,50,100,5000,2,5,0.578454,5000,0,0,0,0,8643.733,8643.733,2026-11-04 11:06:00
105,97,contention_comparison_v1,Stress,baseline,Medium,50,100,5000,3,5,0.748411,5000,0,0,0,0,6680.820,6680.820,2026-11-04 11:06:01
106,99,contention_comparison_v1,Stress,optimistic,Medium,50,100,5000,3,5,0.596379,4252,748,0,0,0,8383.927,7129.691,2026-11-04 11:06:03


## 14. Average the repeated runs


In [14]:
summary_df = (
    runs_df
    .groupby(
        [
            "workload",
            "contention",
            "strategy",
            "thread_count",
            "operations_per_thread",
            "total_operations",
            "copy_count"
        ],
        as_index=False
    )
    .agg(
        avg_runtime_seconds=("runtime_seconds", "mean"),
        avg_successes=("successes", "mean"),
        avg_conflicts=("conflicts", "mean"),
        avg_deadlocks=("deadlocks", "mean"),
        avg_timeouts=("timeouts", "mean"),
        avg_errors=("errors", "mean"),
        avg_attempt_throughput=("attempt_throughput", "mean"),
        avg_successful_throughput=("successful_throughput", "mean")
    )
)

summary_df


,workload,contention,strategy,thread_count,operations_per_thread,total_operations,copy_count,avg_runtime_seconds,avg_successes,avg_conflicts,avg_deadlocks,avg_timeouts,avg_errors,avg_attempt_throughput,avg_successful_throughput
0,High,High,baseline,50,20,1000,1,0.190251,1000.000000,0.000000,0.0,0.0,0.0,5343.567333,5343.567333
1,High,High,optimistic,50,20,1000,1,0.126866,24.666667,975.333333,0.0,0.0,0.0,7958.335667,196.687667
2,High,High,pessimistic,50,20,1000,1,0.132412,1000.000000,0.000000,0.0,0.0,0.0,7631.473667,7631.473667
3,High,Low,baseline,50,20,1000,25,0.162040,1000.000000,0.000000,0.0,0.0,0.0,6195.873333,6195.873333
4,High,Low,optimistic,50,20,1000,25,0.140452,937.000000,63.000000,0.0,0.0,0.0,7191.081000,6760.386000
5,High,Low,pessimistic,50,20,1000,25,0.120339,1000.000000,0.000000,0.0,0.0,0.0,8310.189667,8310.189667
6,High,Medium,baseline,50,20,1000,5,0.177846,1000.000000,0.000000,0.0,0.0,0.0,5626.346667,5626.346667
7,High,Medium,optimistic,50,20,1000,5,0.187121,712.333333,287.666667,0.0,0.0,0.0,5874.616000,4302.924000
8,High,Medium,pessimistic,50,20,1000,5,0.169982,1000.000000,0.000000,0.0,0.0,0.0,6325.314667,6325.314667
9,Low,High,baseline,5,5,25,1,0.007566,25.000000,0.000000,0.0,0.0,0.0,3317.934000,3317.934000


## 15. What this notebook now tests

This version automatically tests three contention levels:

- **High contention:** 1 shared copy
- **Medium contention:** 5 shared copies
- **Low contention:** 25 shared copies

Each workload is repeated 3 times for baseline, pessimistic, and optimistic strategies.

You can still change:
- `BATCH_NAME`
- `REPEATS`
- workload sizes
- number of shared copies

All runs remain stored in `library_results.test_run`.


# 16. Extreme Read-Heavy / Very Low-Contention Test

The systematic tests above focus mainly on concurrent update workloads. As a final edge-case test, this section checks a workload where **optimistic concurrency has conditions that should favor it**:

- 50 concurrent threads
- 500 operations per thread
- 25,000 total operations
- 98% reads
- 2% writes
- a deliberately enlarged inventory of **15,000 book copies**

This is intentionally a different workload from the checkout-heavy tests above. The goal is to test whether optimistic concurrency can become faster when contention is extremely low and most operations are reads.

Before each strategy, the notebook rebuilds `library_sim` and then seeds the same 15,000-copy inventory. This keeps the test repeatable and removes the need to run a separate SQL file manually.

The same generated operation plan is used for pessimistic and optimistic tests within each repeat so the comparison is fair.


## 16.1 Extreme-case settings


In [15]:
import random

EXTREME_BATCH_NAME = "extreme_read_heavy_v1"
EXTREME_REPEATS = 3

EXTREME_THREADS = 50
EXTREME_OPERATIONS_PER_THREAD = 500
EXTREME_READ_PROBABILITY = 0.98

EXTREME_TOTAL_OPERATIONS = (
    EXTREME_THREADS * EXTREME_OPERATIONS_PER_THREAD
)

print(f"Threads: {EXTREME_THREADS}")
print(f"Operations per thread: {EXTREME_OPERATIONS_PER_THREAD}")
print(f"Total operations: {EXTREME_TOTAL_OPERATIONS:,}")
print(f"Expected reads: {EXTREME_READ_PROBABILITY:.0%}")
print(f"Expected writes: {1 - EXTREME_READ_PROBABILITY:.0%}")


Threads: 50
Operations per thread: 500
Total operations: 25,000
Expected reads: 98%
Expected writes: 2%


## 16.2 Create a permanent table for the extreme-case results

These results are stored separately from `test_run` because this experiment mixes reads and writes.


In [16]:
def create_extreme_results_table():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("CREATE SCHEMA IF NOT EXISTS `library_results`")

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS `library_results`.`extreme_case_run` (
            run_id INT NOT NULL AUTO_INCREMENT,
            batch_name VARCHAR(100) NOT NULL,
            strategy VARCHAR(20) NOT NULL,
            repeat_number INT NOT NULL,
            thread_count INT NOT NULL,
            operations_per_thread INT NOT NULL,
            total_operations INT NOT NULL,
            copy_count INT NOT NULL,
            read_percentage DECIMAL(6,3),
            write_percentage DECIMAL(6,3),
            runtime_seconds DECIMAL(14,6),
            read_operations INT,
            write_operations INT,
            successful_operations INT,
            conflicts INT,
            deadlocks INT,
            timeouts INT,
            errors INT,
            attempt_throughput DECIMAL(14,3),
            successful_throughput DECIMAL(14,3),
            run_date DATETIME DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (run_id)
        )
    """)

    conn.commit()
    cursor.close()
    conn.close()

create_extreme_results_table()
print("Extreme-case results table ready.")


Extreme-case results table ready.


## 16.3 Seed a 15,000-copy inventory

This recreates the large-inventory setup from the team member's SQL file directly inside the notebook.

It runs **after** `library_sim` is rebuilt and before each extreme-case strategy, so both pessimistic and optimistic tests receive the same 15,000-copy environment.


In [17]:
EXTREME_COPY_COUNT = 15000

def seed_large_inventory():
    conn = get_connection("library_sim")
    cursor = conn.cursor()

    # Remove loans first because they reference book_copy.
    cursor.execute("DELETE FROM loan")
    cursor.execute("DELETE FROM book_copy")

    insert_sql = """
        INSERT INTO book_copy
        (
            copy_id,
            acquisition_date,
            `condition`,
            `status`,
            Book_book_id,
            version
        )
        VALUES (%s, %s, %s, %s, %s, %s)
    """

    rows = [
        (
            i,
            "2026-01-01",
            "Good",
            "Available",
            1,
            0
        )
        for i in range(1, EXTREME_COPY_COUNT + 1)
    ]

    cursor.executemany(
        insert_sql,
        rows
    )

    conn.commit()
    cursor.close()
    conn.close()


def get_extreme_copy_ids():
    # The seeded inventory is deterministic: copy IDs 1 through 15,000.
    return list(range(1, EXTREME_COPY_COUNT + 1))


def build_extreme_workload(copy_ids, repeat_number):
    rng = random.Random(20261104 + repeat_number)

    plans = []

    for thread_id in range(EXTREME_THREADS):
        thread_plan = []

        for _ in range(EXTREME_OPERATIONS_PER_THREAD):
            copy_id = rng.choice(copy_ids)

            is_write = (
                rng.random() >= EXTREME_READ_PROBABILITY
            )

            thread_plan.append(
                (copy_id, is_write)
            )

        plans.append(thread_plan)

    return plans


## 16.4 Run one strategy against the same workload plan

For a **pessimistic read**, the row is read with `FOR UPDATE`, so the row is locked and the transaction is committed.

For an **optimistic read**, the row is read normally without a row lock. The read-only path does not perform an explicit commit after every read, which removes some overhead in this intentionally read-heavy best-case scenario.

Writes still use the same `checkout_pessimistic` and `checkout_optimistic` procedures defined earlier in this notebook.


In [18]:
def run_extreme_strategy(strategy, workload_plans):
    metrics = {
        "completed_reads": 0,
        "read_operations": 0,
        "write_operations": 0,
        "errors": 0,
        "deadlocks": 0,
        "timeouts": 0
    }

    def worker(thread_id):
        conn = get_connection("library_sim")
        cursor = conn.cursor()

        for copy_id, is_write in workload_plans[thread_id]:
            with metric_lock:
                if is_write:
                    metrics["write_operations"] += 1
                else:
                    metrics["read_operations"] += 1

            try:
                if is_write:
                    if strategy == "pessimistic":
                        cursor.callproc(
                            "checkout_pessimistic",
                            [copy_id]
                        )
                    else:
                        cursor.callproc(
                            "checkout_optimistic",
                            [
                                copy_id,
                                thread_id,
                                "READ COMMITTED"
                            ]
                        )

                    conn.commit()

                else:
                    if strategy == "pessimistic":
                        cursor.execute(
                            """
                            SELECT status
                            FROM book_copy
                            WHERE copy_id = %s
                            FOR UPDATE
                            """,
                            (copy_id,)
                        )

                        cursor.fetchall()
                        conn.commit()

                    else:
                        cursor.execute(
                            """
                            SELECT status, version
                            FROM book_copy
                            WHERE copy_id = %s
                            """,
                            (copy_id,)
                        )

                        cursor.fetchall()

                    with metric_lock:
                        metrics["completed_reads"] += 1

            except Exception as e:
                error_text = str(e)

                with metric_lock:
                    metrics["errors"] += 1

                    if "1205" in error_text:
                        metrics["timeouts"] += 1

                    if "1213" in error_text:
                        metrics["deadlocks"] += 1

                try:
                    conn.rollback()
                except:
                    pass

        cursor.close()
        conn.close()

    threads = []

    start_time = time.perf_counter()

    for thread_id in range(EXTREME_THREADS):
        thread = threading.Thread(
            target=worker,
            args=(thread_id,)
        )

        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    runtime = time.perf_counter() - start_time

    return runtime, metrics


## 16.5 Save the extreme-case result


In [19]:
def save_extreme_result(
    strategy,
    repeat_number,
    copy_count,
    runtime,
    metrics,
    write_successes,
    conflicts
):
    successful_operations = (
        metrics["completed_reads"]
        + write_successes
    )

    attempt_throughput = (
        EXTREME_TOTAL_OPERATIONS / runtime
        if runtime > 0
        else 0
    )

    successful_throughput = (
        successful_operations / runtime
        if runtime > 0
        else 0
    )

    read_percentage = (
        metrics["read_operations"]
        / EXTREME_TOTAL_OPERATIONS
        * 100
    )

    write_percentage = (
        metrics["write_operations"]
        / EXTREME_TOTAL_OPERATIONS
        * 100
    )

    conn = get_connection("library_results")
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO extreme_case_run
        (
            batch_name,
            strategy,
            repeat_number,
            thread_count,
            operations_per_thread,
            total_operations,
            copy_count,
            read_percentage,
            write_percentage,
            runtime_seconds,
            read_operations,
            write_operations,
            successful_operations,
            conflicts,
            deadlocks,
            timeouts,
            errors,
            attempt_throughput,
            successful_throughput
        )
        VALUES
        (
            %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s,
            %s, %s, %s, %s
        )
    """, (
        EXTREME_BATCH_NAME,
        strategy,
        repeat_number,
        EXTREME_THREADS,
        EXTREME_OPERATIONS_PER_THREAD,
        EXTREME_TOTAL_OPERATIONS,
        copy_count,
        read_percentage,
        write_percentage,
        runtime,
        metrics["read_operations"],
        metrics["write_operations"],
        successful_operations,
        conflicts,
        metrics["deadlocks"],
        metrics["timeouts"],
        metrics["errors"],
        attempt_throughput,
        successful_throughput
    ))

    conn.commit()
    cursor.close()
    conn.close()


## 16.6 Run the extreme read-heavy comparison

Each repeat uses the **same exact workload plan** for pessimistic and optimistic locking.

`library_sim` is rebuilt before each strategy, just like the systematic tests above.


In [20]:
for repeat in range(1, EXTREME_REPEATS + 1):
    print("\n" + "=" * 80)
    print(
        f"EXTREME READ-HEAVY TEST | "
        f"Repeat {repeat} of {EXTREME_REPEATS}"
    )
    print("=" * 80)

    # Build the deterministic workload plan once for this repeat.
    copy_ids = get_extreme_copy_ids()

    workload_plans = build_extreme_workload(
        copy_ids,
        repeat
    )

    actual_reads = sum(
        1
        for plan in workload_plans
        for _, is_write in plan
        if not is_write
    )

    actual_writes = (
        EXTREME_TOTAL_OPERATIONS - actual_reads
    )

    print(
        f"Seeded copies: {EXTREME_COPY_COUNT:,} | "
        f"Reads: {actual_reads:,} | "
        f"Writes: {actual_writes:,}"
    )

    # -----------------------------
    # PESSIMISTIC
    # -----------------------------
    print("  Pessimistic...")

    rebuild_library_sim()
    create_checkout_procedures()
    seed_large_inventory()

    pess_time, pess_metrics = run_extreme_strategy(
        "pessimistic",
        workload_plans
    )

    pess_write_successes, pess_conflicts = (
        get_db_metrics("pessimistic")
    )

    save_extreme_result(
        "pessimistic",
        repeat,
        EXTREME_COPY_COUNT,
        pess_time,
        pess_metrics,
        pess_write_successes,
        pess_conflicts
    )

    # -----------------------------
    # OPTIMISTIC
    # -----------------------------
    print("  Optimistic...")

    rebuild_library_sim()
    create_checkout_procedures()
    seed_large_inventory()

    opt_time, opt_metrics = run_extreme_strategy(
        "optimistic",
        workload_plans
    )

    opt_write_successes, opt_conflicts = (
        get_db_metrics("optimistic")
    )

    save_extreme_result(
        "optimistic",
        repeat,
        EXTREME_COPY_COUNT,
        opt_time,
        opt_metrics,
        opt_write_successes,
        opt_conflicts
    )

print("\nExtreme read-heavy testing finished.")



EXTREME READ-HEAVY TEST | Repeat 1 of 3
Seeded copies: 15,000 | Reads: 24,519 | Writes: 481
  Pessimistic...
  Optimistic...

EXTREME READ-HEAVY TEST | Repeat 2 of 3
Seeded copies: 15,000 | Reads: 24,501 | Writes: 499
  Pessimistic...
  Optimistic...

EXTREME READ-HEAVY TEST | Repeat 3 of 3
Seeded copies: 15,000 | Reads: 24,498 | Writes: 502
  Pessimistic...
  Optimistic...

Extreme read-heavy testing finished.


## 16.7 Extreme-case averages

This final table shows whether optimistic concurrency becomes faster under the intentionally favorable **very low-contention, read-heavy** workload.

This result should be interpreted together with the systematic tests above, not as a replacement for them.


In [21]:
conn = get_connection("library_results")

extreme_runs_df = pd.read_sql(
    """
    SELECT *
    FROM extreme_case_run
    WHERE batch_name = %s
    ORDER BY repeat_number, strategy
    """,
    conn,
    params=(EXTREME_BATCH_NAME,)
)

conn.close()

extreme_runs_df


/var/folders/qy/yl4hyv4s5kz52wg1qlh36y_w0000gn/T/ipykernel_7299/1124802689.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  extreme_runs_df = pd.read_sql(


,run_id,batch_name,strategy,repeat_number,thread_count,operations_per_thread,total_operations,copy_count,read_percentage,write_percentage,...,read_operations,write_operations,successful_operations,conflicts,deadlocks,timeouts,errors,attempt_throughput,successful_throughput,run_date
0,2,extreme_read_heavy_v1,optimistic,1,50,500,25000,15000,98.076,1.924,...,24519,481,24999,1,0,0,0,16939.016,16938.339,2026-11-04 11:06:13
1,1,extreme_read_heavy_v1,pessimistic,1,50,500,25000,15000,98.076,1.924,...,24519,481,25000,0,0,0,0,14570.924,14570.924,2026-11-04 11:06:12
2,4,extreme_read_heavy_v1,optimistic,2,50,500,25000,15000,98.004,1.996,...,24501,499,25000,0,0,0,0,15822.945,15822.945,2026-11-04 11:06:17
3,3,extreme_read_heavy_v1,pessimistic,2,50,500,25000,15000,98.004,1.996,...,24501,499,25000,0,0,0,0,13850.370,13850.370,2026-11-04 11:06:15
4,6,extreme_read_heavy_v1,optimistic,3,50,500,25000,15000,97.992,2.008,...,24498,502,25000,0,0,0,0,16057.550,16057.550,2026-11-04 11:06:21
5,5,extreme_read_heavy_v1,pessimistic,3,50,500,25000,15000,97.992,2.008,...,24498,502,25000,0,0,0,0,13093.987,13093.987,2026-11-04 11:06:19


In [22]:
extreme_summary_df = (
    extreme_runs_df
    .groupby(
        [
            "strategy",
            "thread_count",
            "operations_per_thread",
            "total_operations",
            "copy_count"
        ],
        as_index=False
    )
    .agg(
        avg_runtime_seconds=("runtime_seconds", "mean"),
        avg_reads=("read_operations", "mean"),
        avg_writes=("write_operations", "mean"),
        avg_successful_operations=("successful_operations", "mean"),
        avg_conflicts=("conflicts", "mean"),
        avg_errors=("errors", "mean"),
        avg_attempt_throughput=("attempt_throughput", "mean"),
        avg_successful_throughput=("successful_throughput", "mean")
    )
)

extreme_summary_df


,strategy,thread_count,operations_per_thread,total_operations,copy_count,avg_runtime_seconds,avg_reads,avg_writes,avg_successful_operations,avg_conflicts,avg_errors,avg_attempt_throughput,avg_successful_throughput
0,optimistic,50,500,25000,15000,1.537589,24506.0,494.0,24999.666667,0.333333,0.0,16273.170333,16272.944667
1,pessimistic,50,500,25000,15000,1.810008,24506.0,494.0,25000.000000,0.000000,0.0,13838.427000,13838.427000


## 16.8 Final interpretation

The systematic experiments above test how pessimistic and optimistic locking behave as **write contention changes**.

This final extreme case deliberately creates a much larger inventory of **15,000 book copies** and uses a **98% read / 2% write** workload. That spreads activity across many rows and creates a scenario where optimistic concurrency has a chance to show its advantage.

If optimistic locking is faster here, that supports the idea that neither concurrency strategy is always best:

- **Pessimistic locking** is better suited for concentrated, write-heavy contention because it prevents repeated conflicting updates.
- **Optimistic concurrency** can be more efficient when contention is very low and most operations are reads because it avoids row-locking overhead on the read path.

The conclusion should therefore be based on the workload: the best strategy depends on how often transactions write and how often users compete for the same rows.
